# Importing Libraries

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Create MySQL Connection


In [3]:
username = "root"
password = "affanimam%40123"
host = "localhost:3306"
database = "ecommerce_analytics"
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# Load Cleaned CSV Files

In [4]:
data_path = "../cleaned_data/"

orders = pd.read_csv(data_path + "orders_cleaned.csv")
customers = pd.read_csv(data_path + "customers_cleaned.csv")
order_items = pd.read_csv(data_path + "order_items_cleaned.csv")
order_payments = pd.read_csv(data_path + "order_payments_cleaned.csv")
order_reviews = pd.read_csv(data_path + "order_reviews_cleaned.csv")
products = pd.read_csv(data_path + "products_cleaned.csv")
sellers = pd.read_csv(data_path + "sellers_cleaned.csv")
geolocation = pd.read_csv(data_path + "geolocation_cleaned.csv")
category_translation = pd.read_csv(data_path + "category_translation_cleaned.csv")

# Uploading DataFrames to MySQL

In [5]:
customers.to_sql("customers", con=engine, if_exists="replace", index=False)
orders.to_sql("orders", con=engine, if_exists="replace", index=False)
products.to_sql("products", con=engine, if_exists="replace", index=False)
sellers.to_sql("sellers", con=engine, if_exists="replace", index=False)
order_items.to_sql("order_items", con=engine, if_exists="replace", index=False)
order_payments.to_sql("order_payments", con=engine, if_exists="replace", index=False)
order_reviews.to_sql("order_reviews", con=engine, if_exists="replace", index=False)
geolocation.to_sql("geolocation", con=engine, if_exists="replace", index=False)
category_translation.to_sql("category_translation", con=engine, if_exists="replace", index=False)

print("All tables uploaded successfully ")

All tables uploaded successfully 


# Verifying Tables in MySQL

In [6]:
pd.read_sql("SELECT COUNT(*) FROM orders", engine)

,COUNT(*)
0,99441


# Basic Business Queries

### Total orders

In [7]:
query1 = """
SELECT COUNT(*) AS total_orders
FROM orders;
"""

df_total_orders = pd.read_sql(query1, engine)
df_total_orders

,total_orders
0,99441


### Total Revenue

In [8]:
query2 = """
SELECT ROUND(SUM(payment_value),2) AS total_revenue
FROM order_payments;
"""

df_total_revenue = pd.read_sql(query2, engine)
df_total_revenue

,total_revenue
0,16008872.12


### Total Unique Customers

In [9]:
query3 = """
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers;
"""

df_unique_customers = pd.read_sql(query3, engine)
df_unique_customers

,unique_customers
0,96096


### Orders by Status

In [10]:
query4 = """
SELECT order_status, COUNT(*) AS total_orders
FROM orders
GROUP BY order_status;
"""

df_order_status = pd.read_sql(query4, engine)
df_order_status

,order_status,total_orders
0,delivered,96478
1,invoiced,314
2,shipped,1107
3,processing,301
4,unavailable,609
5,canceled,625
6,created,5
7,approved,2


# Intermediate Queries

### Monthly Revenue Trend

In [12]:
query5 = """
SELECT DATE_FORMAT(o.order_purchase_timestamp,'%%Y-%%m') AS month,
       ROUND(SUM(p.payment_value),2) AS revenue
FROM orders o
JOIN order_payments p ON o.order_id = p.order_id
GROUP BY month
ORDER BY month;
"""

df_monthly_revenue = pd.read_sql(query5, engine)
df_monthly_revenue

,month,revenue
0,2016-09,252.24
1,2016-10,59090.48
2,2016-12,19.62
3,2017-01,138488.04
4,2017-02,291908.01
5,2017-03,449863.60
6,2017-04,417788.03
7,2017-05,592918.82
8,2017-06,511276.38
9,2017-07,592382.92


### Top 10 States by Orders

In [13]:
query6 = """
SELECT c.customer_state,
       COUNT(o.order_id) AS total_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10;
"""

df_top_states = pd.read_sql(query6, engine)
df_top_states

,customer_state,total_orders
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


### Average Order Value

In [14]:
query7 = """
SELECT ROUND(AVG(payment_value),2) AS avg_order_value
FROM order_payments;
"""

df_aov = pd.read_sql(query7, engine)
df_aov

,avg_order_value
0,154.1


### Payment Method Distribution

In [15]:
query8 = """
SELECT payment_type,
       COUNT(*) AS count,
       ROUND(SUM(payment_value),2) AS total_value
FROM order_payments
GROUP BY payment_type;
"""

df_payment = pd.read_sql(query8, engine)
df_payment

,payment_type,count,total_value
0,credit_card,76795,12542084.19
1,boleto,19784,2869361.27
2,voucher,5775,379436.87
3,debit_card,1529,217989.79
4,not_defined,3,0.00


# Advanced Queries

### Delivery Performance

In [16]:
query9 = """
SELECT 
    ROUND(AVG(delivery_delay_days),2) AS avg_delay,
    ROUND(AVG(on_time_delivery)*100,2) AS on_time_delivery_percentage
FROM orders;
"""

df_delivery = pd.read_sql(query9, engine)
df_delivery

,avg_delay,on_time_delivery_percentage
0,-11.88,90.45


### Top Product Categories by Revenue

In [17]:
query10 = """
SELECT ct.product_category_name_english,
       ROUND(SUM(oi.price),2) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct 
ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 10;
"""

df_top_categories = pd.read_sql(query10, engine)
df_top_categories

,product_category_name_english,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


### Repeat Customers

In [18]:
query11 = """
SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT customer_unique_id
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY customer_unique_id
    HAVING COUNT(o.order_id) > 1
) AS repeat_table;
"""

df_repeat_customers = pd.read_sql(query11, engine)
df_repeat_customers

,repeat_customers
0,2997


### Revenue by Month & Category

In [20]:
query12 = """
SELECT DATE_FORMAT(o.order_purchase_timestamp,'%%Y-%%m') AS month,
       ct.product_category_name_english,
       ROUND(SUM(oi.price),2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct 
ON p.product_category_name = ct.product_category_name
GROUP BY month, ct.product_category_name_english
ORDER BY month;
"""

df_month_category = pd.read_sql(query12, engine)
df_month_category

,month,product_category_name_english,revenue
0,2016-09,furniture_decor,72.89
1,2016-09,health_beauty,134.97
2,2016-09,telephony,59.50
3,2016-10,air_conditioning,1707.09
4,2016-10,audio,156.99
...,...,...,...
1248,2018-08,stationery,15563.65
1249,2018-08,telephony,37541.89
1250,2018-08,toys,17753.05
1251,2018-08,watches_gifts,72277.06


# Savings important for visualization

In [21]:
df_monthly_revenue.to_csv("../cleaned_data/monthly_revenue.csv", index=False)
df_top_states.to_csv("../cleaned_data/top_states.csv", index=False)
df_top_categories.to_csv("../cleaned_data/top_categories.csv", index=False)
df_payment.to_csv("../cleaned_data/payment_distribution.csv", index=False)